In [ ]:
!pip install --upgrade google-cloud-aiplatform

In [ ]:
!pip install docling

In [ ]:
import google.auth
credentials, PROJECT_ID = google.auth.default()
PROJECT_ID

In [ ]:
import os
REGION = "asia-northeast3"
PACKAGE_PATH = "package"
REPO_NAME = "custom-inference-gpu"
BASE_CONTAINER = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO_NAME}/docling-cu128"
SERVICE_ACCOUNT = f"{PROJECT_ID}@{PROJECT_ID}.iam.gserviceaccount.com"   #SA to access gcs
BUCKET_NAME = f"{PROJECT_ID}-vai"
MODEL_PATH = f"gs://{BUCKET_NAME}/"

os.system(f"mkdir -p {PACKAGE_PATH}")

In [ ]:
%%writefile Dockerfile

# For building docling running on Vertex AI endpoint
# 1. NVIDIA CUDA 12.8 런타임 베이스 이미지 (Ubuntu 22.04 기준)
FROM nvidia/cuda:12.8.0-cudnn-runtime-ubuntu22.04

# 대화형 프롬프트 방지 및 환경변수 설정 (ENV 블록 내 중간 주석 제거)
ENV DEBIAN_FRONTEND=noninteractive \
    PYTHONUNBUFFERED=1 \
    PYTHONDONTWRITEBYTECODE=1 \
    PIP_NO_CACHE_DIR=1 \
    HF_HOME=/opt/docling_cache/huggingface \
    TORCH_HOME=/opt/docling_cache/torch \
    EASYOCR_MODULE_PATH=/opt/docling_cache/easyocr \
    DOCLING_ARTIFACTS_PATH=/opt/docling_cache/models \
    OMP_NUM_THREADS=4 \
    VERTEX_CPR_MAX_WORKERS=1

# 2. Docling 동작에 필요한 모든 OS 레벨 필수 패키지 설치
# - python-is-python3: /usr/bin/python -> python3 공식 심볼릭 링크 패키지
RUN apt-get update && apt-get install -y --no-install-recommends \
    python3 \
    python-is-python3 \
    python3-pip \
    python3-venv \
    python3-dev \
    build-essential \
    git \
    curl \
    ca-certificates \
    libgl1 \
    libglib2.0-0 \
    libsm6 \
    libxext6 \
    libxrender1 \
    libgomp1 \
    tesseract-ocr \
    tesseract-ocr-eng \
    tesseract-ocr-kor \
    tesseract-ocr-osd \
    libtesseract-dev \
    poppler-utils \
    libreoffice \
    ffmpeg \
    libsndfile1 \
    && rm -rf /var/lib/apt/lists/*

# 캐시 디렉토리 생성 및 권한 부여 (Vertex AI 비루트 실행 대비 777 권한 권장)
RUN mkdir -p ${HF_HOME} ${TORCH_HOME} ${EASYOCR_MODULE_PATH} ${DOCLING_ARTIFACTS_PATH} && \
    chmod -R 777 /opt/docling_cache

# 3. pip 업그레이드 및 PyTorch (CUDA 12.8 전용 휠) 선설치
# (python-is-python3 덕분에 이제부터 바로 python 명령 사용 가능)
RUN python -m pip install --upgrade pip setuptools wheel && \
    python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

# 4. Docling 및 모든 선택적/확장 의존성 패키지 설치
RUN python -m pip install \
    "docling[vlm,easyocr,rapidocr,tesserocr]" \
    onnxruntime-gpu \
    accelerate \
    bitsandbytes

# 5. 클래스 Cold Start 방지를 위한 모델 가중치 사전 다운로드 (Pre-warming)
RUN docling-tools models download -o ${DOCLING_ARTIFACTS_PATH}

In [ ]:
os.system(f"gcloud builds submit --region={REGION} --tag={BASE_CONTAINER}") #Additionally you can specify build machine type for faster build

In [ ]:
%%writefile {PACKAGE_PATH}/requirements.txt

google-cloud-storage

In [ ]:
%%writefile {PACKAGE_PATH}/DoclingPredictor.py

import logging
import time
from google.cloud.aiplatform.prediction.predictor import Predictor
from docling.document_converter import DocumentConverter

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

class DoclingPredictor(Predictor):
    def __init__(self):
        return

    def load(self, artifacts_uri: str) -> None:
        # Load model
        logger.info(f"Starting predictor using {artifacts_uri}")
        self.converter = DocumentConverter()
        logger.info('Predictor loaded successfully')

    def predict(self, prediction_input):
        start_time = time.time()
        logger.info(f"--- input: {prediction_input['instances']}")
        doc = None
        try:
            doc = self.converter.convert(prediction_input['instances'][0]).document
        except Exception as e:
            logger.info(e)
        logger.info("--- %s seconds ---" % (time.time() - start_time))
        return {"predictions": [doc.export_to_markdown()]}

In [ ]:
from package.DoclingPredictor import DoclingPredictor
from google.cloud.aiplatform.prediction import LocalModel
#add custom predictor
vai_serving_container_uri = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO_NAME}/doclingpredictor" #Must be lower case

#must secure sufficient space
local_model = LocalModel.build_cpr_model(
    src_dir=PACKAGE_PATH,
    output_image_uri=vai_serving_container_uri,
    predictor=DoclingPredictor,
    requirements_path=f"{PACKAGE_PATH}/requirements.txt",
    #extra_packages=["deploy_package/custom_package.tar.gz"]
    base_image=f"{BASE_CONTAINER}",
    no_cache = False
)

In [ ]:
import json
import sys
import logging
logging.basicConfig(level=logging.INFO, stream=sys.stderr)

local_endpoint = local_model.deploy_to_local_endpoint(artifact_uri=MODEL_PATH, gpu_count=0) #Set -1 if using GPU
try:
    local_endpoint.serve()
finally:
    #Run below if need to get container log
    print(local_endpoint.container.logs().decode("utf-8").strip(), sep="\n")

In [ ]:
request = {
        "instances" : [
            "https://arxiv.org/pdf/2206.01062",
        ]
    }
predict_response = local_endpoint.predict(
        request=json.dumps(request),
        headers={"Content-Type": "application/json"},
    )
predict_response.json()

In [ ]:
print(local_endpoint.container.logs().decode("utf-8").strip(), sep="\n")

In [ ]:
local_endpoint.stop()

In [ ]:
#Push locally built image to artifact registry for deploy
local_model.push_image()

In [ ]:
#Make model be used for Model repository
from google.cloud import aiplatform
model = aiplatform.Model.upload(
    location=REGION,
    display_name = "DoclingPredictor",
    local_model = local_model,
    artifact_uri = MODEL_PATH,
    #parent_model = prev_model.resource_name,
    #is_default_version=True,
    serving_container_environment_variables={
        # Optional env var so that `uvicorn` only runs the model in 1 worker
        #"VERTEX_CPR_MAX_WORKERS": 4,
    },
)

In [ ]:
#Create endpoint for model hosting
remote_endpoint = aiplatform.Endpoint.create(
    display_name=f"DoclingPredictor test endpoint",
    #labels={"sample-key": "sample-value"},
    location=REGION,
    dedicated_endpoint_enabled=True,
)

In [ ]:
#Deploy model under the endpoint
remote_endpoint.deploy(
    model=model,
    machine_type="g2-standard-4",
    #tpu_topology=None,
    min_replica_count=1,
    max_replica_count=1,
    #service_account=SERVICE_ACCOUNT,
    #traffic_percentage=50
    #traffic_split={'a':50, 'b':50}
    #Configs for GPU
    accelerator_type="NVIDIA_L4",
    accelerator_count=1,
    #deploy_request_timeout=DEPLOY_TIMEOUT
)

In [ ]:
import json
import time
from google.cloud import aiplatform

#Create endpoint class
endpoint = aiplatform.Endpoint("projects/sandbox-373102/locations/asia-northeast3/endpoints/4693834930185043968")

request = {
        "instances" : [
            "https://arxiv.org/pdf/2206.01062",
        ]
    }

In [ ]:
response = endpoint.raw_predict(
        body=json.dumps(request).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        use_dedicated_endpoint=True,  # False for a shared endpoint
    )

response.json()